# Dreamy — Colab T4 训练

**使用前：运行时 → 更改运行时类型 → 硬件加速器选 T4 GPU**

流程：
1. 挂载 Google Drive
2. 拉取本仓库代码
3. 安装依赖（自动选 CUDA wheel）
4. 冒烟测试（确认 T4 的 sm_75 kernel 可用）
5. 开始训练，checkpoint 写回 Drive

## 0. 确认 GPU 已启用

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
print('\n如果不是 T4，请到 运行时 → 更改运行时类型 里切换')

## 1. 挂载 Google Drive

训练数据和 checkpoint 都在 Drive 上，Colab 断线后可以接着跑。

在 Drive 里建好目录结构：
```
MyDrive/dreamy/
├── data/          <- 训练数据放这里（比如 tokens.bin）
└── runs/          <- checkpoint 和日志，脚本会自动创建
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/dreamy/data'
OUT_DIR  = '/content/drive/MyDrive/dreamy/runs'

print('data 目录:', DATA_DIR, '->', '存在' if os.path.isdir(DATA_DIR) else '不存在')
if os.path.isdir(DATA_DIR):
    print('内容:', os.listdir(DATA_DIR)[:20])

## 2. 拉取代码

**注意：这个仓库是 public 才能直接 clone。若是私有仓库，需要 Personal Access Token**（见下方注释）。

In [ ]:
REPO = 'https://github.com/qsustudio/Dreamy.git'

if not os.path.isdir('/content/Dreamy'):
    !git clone $REPO /content/Dreamy
else:
    # 已拉过则更新
    !cd /content/Dreamy && git pull

%cd /content/Dreamy
!ls -la

### （可选）私有仓库：用 PAT 拉取
```python
# 在 https://github.com/settings/tokens 建一个 repo 权限的 token
TOKEN = 'ghp_xxx'
!git clone https://{TOKEN}@github.com/qsustudio/Dreamy.git /content/Dreamy
```
⚠️ 不要把 token 写进 notebook 再提交到仓库。

## 3. 安装依赖

**关键：torch 的版本选择。**

T4 是 Turing 架构（sm_75）。PyTorch 2.15 起不再发布 cu126 wheel，而 CUDA 13 系列要求驱动 ≥580 —— Colab 的 T4 驱动常常低于 580。所以**锁定 cu126 + torch 2.14.x** 最稳。

In [ ]:
# 装 cu126 版 torch（适配 T4 + 常见 Colab 驱动版本）
!pip install -q torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126

# 装项目依赖（x-transformers 等，torch 不在此文件里）
!pip install -q -r requirements.txt

## 4. 冒烟测试

确认 torch 构建真的包含 T4 的 sm_75 kernel，模型能前向+反向。

In [ ]:
!python scripts/smoke_test.py

如果上面 `当前 GPU kernel 可用 (sm_75)` 是 **FAIL**，说明装的 torch 不含 Turing kernel，
请回到第 3 步确认用的是 cu126 版本。

## 5. 开始训练

checkpoint 会写到 `MyDrive/dreamy/runs/checkpoints/last.pt`，Colab 断线后重连运行同一个 cell 加 `--resume auto` 即可续训。

In [ ]:
import os
os.environ['DREAMY_DATA_DIR'] = DATA_DIR
os.environ['DREAMY_OUT_DIR']  = OUT_DIR

# 首次训练
!python src/train.py --max-steps 1000 --batch-size 32

# 断线重连后续训（取消下面注释，注释掉上面那行）
# !python src/train.py --max-steps 1000 --batch-size 32 --resume auto

## 6. 检查输出

Colab 运行时会话结束后本地文件全丢 —— **只有 Drive 上的东西会保留**。

In [ ]:
!ls -lh $OUT_DIR/checkpoints/ 2>/dev/null || echo '还没有 checkpoint'